## Welcome to the ebook2audiobook Google Colab!
## Features
- 🔧 **TTS Engines supported**: XTTSv2, Bark, Fairseq, VITS, Tacotron2, Tortoise, GlowTTS, YourTTS- 📚 **Convert multiple file formats**: .epub, .mobi, .azw3, .fb2, .lrf, .rb, .snb, .tcr, .pdf, .txt, .rtf, .doc, .docx, .html, .odt, .azw, .tiff, .tif, .png, .jpg, .jpeg, .bmp
- 🔍 **OCR scanning** for files with text pages as images
- 🔊 **High-quality text-to-speech** from near realtime to near real voice
- 🗣️ **Optional voice cloning** using your own voice file
- 🌐 **Supports 1158 languages** ([supported languages list](https://dl.fbaipublicfiles.com/mms/tts/all-tts-languages.html))
- 💻 **Low-resource friendly** — runs on **2 GB RAM / 1 GB VRAM (minimum)**
- 🎵 **Audiobook output formats**: mono or stereo aac, flac, mp3, m4b, m4a, mp4, mov, ogg, wav, webm
- 🧠 **SML tags supported** — fine-grained control of breaks, pauses, voice switching and more ([see below](#sml-tags-available))
- 🧩 **Optional custom model** using your own trained model (XTTSv2 only, other on request)
- 🎛️ **Fine-tuned preset models** trained by the E2A Team<br/>
     <i>(Contact us if you need additional fine-tuned models, or if you'd like to share yours to the official preset list)</i>
## Want to run locally for free? ⬇
## [Check out the ebook2audiobook github!](https://github.com/pcg1974/ebook2audiobook)

In [3]:
# @title 💾 Google Drive Persistence Setup

import os
from google.colab import drive

print("1. Mounting Google Drive...")
drive.mount('/content/drive', force_remount=True)

# Updated to your custom path
DRIVE_STORAGE = "/content/drive/MyDrive/Colab/ebook2audiobook"
os.makedirs(f"{DRIVE_STORAGE}/voices", exist_ok=True)
os.makedirs(f"{DRIVE_STORAGE}/audiobooks", exist_ok=True)

# Prepare local directory and create symbolic links
SCRIPT_DIR = "/content/ebook2audiobook"
os.makedirs(SCRIPT_DIR, exist_ok=True)

# Link voices and audiobooks folders to Google Drive
os.system(f"rm -rf {SCRIPT_DIR}/voices && ln -s {DRIVE_STORAGE}/voices {SCRIPT_DIR}/voices")
os.system(f"rm -rf {SCRIPT_DIR}/audiobooks && ln -s {DRIVE_STORAGE}/audiobooks {SCRIPT_DIR}/audiobooks")

print("\n✅ Setup complete! Voices and Audiobooks will automatically save to your Google Drive under:")
print(f"   {DRIVE_STORAGE}")

1. Mounting Google Drive...
Mounted at /content/drive

✅ Setup complete! Voices and Audiobooks will automatically save to your Google Drive under:
   /content/drive/MyDrive/Colab/ebook2audiobook


In [ ]:
# @title 🚀 Run ebook2audiobook!

import os
import subprocess
import time
import shutil
import sysconfig
import sys

# Emojis for logs
CHECK_MARK = "✅"
CROSS_MARK = "❌"

SCRIPT_DIR = "/content/ebook2audiobook"
VENV_DIR = f"{SCRIPT_DIR}/python_env"
VENV_PYTHON = f"{VENV_DIR}/bin/python"

# ── Ensure system temporary directory exists BEFORE anything else runs ─────────
os.makedirs(f"{SCRIPT_DIR}/tmp", exist_ok=True)
os.chmod(f"{SCRIPT_DIR}/tmp", 0o777)

# ── Environment variables ────────────────────────────────────────────────────
os.environ["PYTHONUTF8"] = "1"
os.environ["PYTHONIOENCODING"] = "utf-8"
os.environ["TTS_CACHE"] = f"{SCRIPT_DIR}/models"
os.environ["TESSDATA_PREFIX"] = f"{SCRIPT_DIR}/models/tessdata"
os.environ["TMPDIR"] = "/tmp"  # Use standard /tmp for OS installers like Rustup
os.environ["SCRIPT_MODE"] = "native"  # Direct app.py to run in native environment mode
os.environ["MPLBACKEND"] = "Agg"

def display_loading_bar(total_steps):
    print("\n--- LOADING... Total steps:", total_steps, " ---")

def update_progress(step, total_steps):
    bar_length = 20
    progress_percent = int((step / total_steps) * 100)
    progress_filled = int(bar_length * step / total_steps)
    bar = '=' * progress_filled + '>' + ' ' * max(bar_length - progress_filled - 1, 0)
    print(f"--- PROGRESS: [{bar}] {progress_percent}% ({step}/{total_steps}) ---")

def run_command_with_log(command, description, step_progress, total_step_commands, forced_cwd=None):
    """Runs a shell command and logs progress, outcome and duration."""
    print(f"\n{step_progress}/{total_step_commands}: {description}...")
    start_time = time.time()
    try:
        working_dir = forced_cwd if forced_cwd else (SCRIPT_DIR if os.path.exists(SCRIPT_DIR) else "/content")

        process = subprocess.Popen(command, shell=True, cwd=working_dir,
                                   stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        stdout, stderr = process.communicate()
        duration = f"{time.time() - start_time:.2f}"
        if process.returncode != 0:
            print(f"{CROSS_MARK} Command failed: {description} (Took {duration}s)")
            print(f"   Command: {command}")
            print(f"   Error Output:\n{stderr.decode()}")
            return False
        else:
            update_progress(step_progress, total_step_commands)
            print(f"{CHECK_MARK} {step_progress}/{total_step_commands} completed: {description} (Took {duration}s)")
            return True
    except Exception as e:
        duration = f"{time.time() - start_time:.2f}"
        print(f"{CROSS_MARK} Error during: {description} (Took {duration}s) — {e}")
        return False


# ── Step 1 : OS-level packages ────────────────────────────────────────────────
os_install_commands = [
    ("apt-get update -qq",
     "Update package lists"),
    ("sudo add-apt-repository ppa:deadsnakes/ppa -y && apt-get update -qq",
     "Add PPA for older Python versions"),
    ("apt-get install -y -qq python3.10 python3.10-venv python3.10-distutils",
     "Install Python 3.10 (Required by ebook2audiobook)"),
    ("apt-get install -y -qq libxcb-cursor0 libegl1 libopengl0",
     "Install Calibre display libraries"),
    ("sudo -v && wget -nv -O- https://download.calibre-ebook.com/linux-installer.sh | sudo sh /dev/stdin",
     "Download & install Calibre"),
    ("apt-get install -y -qq ffmpeg",
     "Install ffmpeg"),
    ("apt-get install -y -qq mediainfo",
     "Install mediainfo"),
    ("apt-get install -y -qq nodejs",
     "Install nodejs"),
    ("apt-get install -y -qq espeak-ng",
     "Install espeak-ng"),
    ("apt-get install -y -qq sox",
     "Install sox"),
    ("apt-get install -y -qq tesseract-ocr tesseract-ocr-eng",
     "Install Tesseract OCR + English language pack"),
    ("apt-get install -y -qq mecab libmecab-dev mecab-ipadic-utf8 && touch /etc/mecabrc",
     "Install mecab & create system mecabrc"),
    ("curl -fsSL https://sh.rustup.rs | sh -s -- -y --quiet && "
     "echo 'source $HOME/.cargo/env' >> ~/.bashrc",
     "Install Rust (required by some Python packages)"),
]

# ── Step 2 : Git clone ────────────────────────────────────────────────────────
git_commands = [
    (f"find {SCRIPT_DIR} -maxdepth 1 ! -type l ! -name 'ebook2audiobook' -exec rm -rf {{}} + 2>/dev/null || true",
     "Clean existing repo contents while preserving symlinks"),
    (f"git clone --depth=1 https://github.com/pcg1974/ebook2audiobook.git /content/temp_repo && "
     f"cp -rn /content/temp_repo/* {SCRIPT_DIR}/ && "
     f"cp -rn /content/temp_repo/.* {SCRIPT_DIR}/ 2>/dev/null || true && "
     f"rm -rf /content/temp_repo",
     "Populate repo files into target directory"),
]

# ── Step 3 : Virtual Environment & Python packages ───────────────────────────
pip_commands = [
    (f"python3.10 -m venv --without-pip {VENV_DIR}",
     "Create Python 3.10 virtual environment in python_env"),
    (f"curl -sSL https://bootstrap.pypa.io/get-pip.py | {VENV_PYTHON}",
     "Bootstrap pip inside virtual environment"),
    (f"{VENV_PYTHON} -m pip install -q --upgrade pip setuptools wheel packaging",
     "Upgrade pip / setuptools / wheel inside venv"),
    (f"{VENV_PYTHON} -m pip install -q --upgrade llvmlite numba --only-binary=:all:",
     "Install llvmlite & numba"),
    (f"{VENV_PYTHON} -m pip install -e {SCRIPT_DIR}/ext/py/demucs --no-deps -q",
     "Install local demucs package"),
    (f"sed -i '/ext\\/py\\/demucs/d' {SCRIPT_DIR}/requirements.txt && "
     f"{VENV_PYTHON} -m pip install -q --no-cache-dir -r {SCRIPT_DIR}/requirements.txt",
     "Install Python requirements from requirements.txt"),
    (f"{VENV_PYTHON} -m pip install -q unidic-lite && {VENV_PYTHON} -m pip uninstall -y unidic",
     "Install unidic-lite and remove broken unidic to trigger native fallback"),
]


# ════════════════════════════════════════════════════════════════════════════
# EXECUTION
# ════════════════════════════════════════════════════════════════════════════

os.chdir("/content")

# --- Step 1: OS packages ---
print("\n--- Step 1: OS-Level Installations ---")
print("Installs system packages required by the bash script. (~3-5 min)")
display_loading_bar(len(os_install_commands))
step1_ok = True
for i, (cmd, desc) in enumerate(os_install_commands):
    if not run_command_with_log(cmd, desc, i + 1, len(os_install_commands), forced_cwd="/content"):
        step1_ok = False
        break

cargo_env = os.path.expanduser("~/.cargo/env")
if os.path.exists(cargo_env):
    os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ.get("PATH", "")

if step1_ok:
    print(f"\n{CHECK_MARK} Step 1: OS-Level Installations — Completed Successfully!")
else:
    print(f"\n{CROSS_MARK} Step 1: OS-Level Installations — Warning: Step 1 had issues.")


# --- Step 2: Git Clone ---
print("\n--- Step 2: Git Clone ---")
display_loading_bar(len(git_commands))
step2_ok = True
for i, (cmd, desc) in enumerate(git_commands):
    if not run_command_with_log(cmd, desc, i + 1, len(git_commands), forced_cwd="/content"):
        step2_ok = False
        break

if step2_ok:
    print(f"\n{CHECK_MARK} Step 2: Git Clone — Completed Successfully!")
else:
    print(f"\n{CROSS_MARK} Step 2: Git Clone — Failed. Aborting script.")
    sys.exit(1)


# --- Step 3: Python Virtual Environment & Package Installation ---
print("\n--- Step 3: Virtual Environment Setup & Package Installation ---")
print("Creates python_env and installs Python requirements. (~3-5 min)")
display_loading_bar(len(pip_commands))
step3_ok = True
for i, (cmd, desc) in enumerate(pip_commands):
    if not run_command_with_log(cmd, desc, i + 1, len(pip_commands), forced_cwd=SCRIPT_DIR):
        step3_ok = False
        break

if step3_ok:
    print(f"\n{CHECK_MARK} Step 3: Python Package Installation — Completed Successfully!")
else:
    print(f"\n{CROSS_MARK} Step 3: Python Package Installation — Failed. See errors above.")
    sys.exit(1)


# --- Step 4: Hooks, Directories, and Bug Patches ---
print("\n--- Step 4: Preparing system & fixing source bugs ---")

# 1. Base sitecustomize.py
src = f"{SCRIPT_DIR}/components/sitecustomize.py"
dst_dir = subprocess.check_output([VENV_PYTHON, "-c", "import sysconfig; print(sysconfig.get_paths()['purelib'])"]).decode().strip()
dst = os.path.join(dst_dir, "sitecustomize.py")
if os.path.exists(src):
    shutil.copy2(src, dst)

# 2. Create directories
for d in ["models", "models/tessdata", "tmp", "run", "audiobooks", "ebooks", "voices"]:
    dir_path = f"{SCRIPT_DIR}/{d}"
    if not os.path.islink(dir_path):
        os.makedirs(dir_path, exist_ok=True)
os.chmod(f"{SCRIPT_DIR}/tmp", 0o777)

# 3. Ensure MeCab OS initialization doesn't error out
if not os.path.exists("/etc/mecabrc"):
    os.system("sudo touch /etc/mecabrc")

# 4. Patch device_installer.py
patch_file = f"{SCRIPT_DIR}/lib/classes/device_installer.py"
if os.path.exists(patch_file):
    with open(patch_file, "r", encoding="utf-8") as f:
        code = f.read()

    code = code.replace("tag_ver = _normalize_version(ver_str)", "tag_ver = __import__('packaging.version').version.parse(str(ver_str))")
    code = code.replace("v <= version", "v <= __import__('packaging.version').version.parse(str(version))")

    with open(patch_file, "w", encoding="utf-8") as f:
        f.write(code)
    print(f"{CHECK_MARK} Cleanly patched source code bugs.")


# --- Step 5: Run App ---
print("\n--- Step 5: Launch ebook2audiobook ---")
print("Starting the Gradio web interface with a public share link...")
os.environ["GRADIO_ALLOWED_PATHS"] = "/content/drive/MyDrive/Colab/ebook2audiobook,/content/ebook2audiobook"

try:
    get_ipython().system(
        f"cd {SCRIPT_DIR} && "
        f"GRADIO_ALLOWED_PATHS='/content/drive/MyDrive/Colab/ebook2audiobook,/content/ebook2audiobook' "
        f"VIRTUAL_ENV={VENV_DIR} {VENV_PYTHON} -u app.py --script_mode native --share"
    )
except Exception as e:
    print(f"{CROSS_MARK} Error starting app.py: {e}")


--- Step 1: OS-Level Installations ---
Installs system packages required by the bash script. (~3-5 min)

--- LOADING... Total steps: 13  ---

1/13: Update package lists...
--- PROGRESS: [=>                  ] 7% (1/13) ---
✅ 1/13 completed: Update package lists (Took 5.68s)

2/13: Add PPA for older Python versions...
--- PROGRESS: [===>                ] 15% (2/13) ---
✅ 2/13 completed: Add PPA for older Python versions (Took 12.04s)

3/13: Install Python 3.10 (Required by ebook2audiobook)...
--- PROGRESS: [====>               ] 23% (3/13) ---
✅ 3/13 completed: Install Python 3.10 (Required by ebook2audiobook) (Took 5.96s)

4/13: Install Calibre display libraries...
--- PROGRESS: [======>             ] 30% (4/13) ---
✅ 4/13 completed: Install Calibre display libraries (Took 5.17s)

5/13: Download & install Calibre...
--- PROGRESS: [=======>            ] 38% (5/13) ---
✅ 5/13 completed: Download & install Calibre (Took 35.56s)

6/13: Install ffmpeg...
--- PROGRESS: [=========>          

In [ ]:
# @title 🚁 Relaunch app from existing virtual environment
!cd /content/ebook2audiobook && \
 GRADIO_ALLOWED_PATHS='/content/drive/MyDrive/Colab/ebook2audiobook,/content/ebook2audiobook' \
 VIRTUAL_ENV=/content/ebook2audiobook/python_env \
 /content/ebook2audiobook/python_env/bin/python -u app.py --script_mode native --share

In [7]:
# @title 📦 Create Environment Snapshot to Google Drive
import os

DRIVE_STORAGE = "/content/drive/MyDrive/Colab/ebook2audiobook"
BACKUP_FILE = f"{DRIVE_STORAGE}/env_backup.tar.gz"

print("Packing virtual environment and unidic dict into Google Drive... (~1-2 min)")
# Archive python_env and unidic downloads
os.system(f"tar -czf {BACKUP_FILE} -C /content/ebook2audiobook python_env models")

print(f"✅ Backup created successfully at:\n   {BACKUP_FILE}")

Packing virtual environment and unidic dict into Google Drive... (~1-2 min)
✅ Backup created successfully at:
   /content/drive/MyDrive/Colab/ebook2audiobook/env_backup.tar.gz


In [ ]:
# @title 🚀 Fast Startup for ebook2audiobook (Colab / Drive persistent)

import os
import subprocess
import time
import shutil
import sys

# Emojis for logs
CHECK_MARK = "✅"
CROSS_MARK = "❌"

SCRIPT_DIR = "/content/ebook2audiobook"
VENV_DIR = f"{SCRIPT_DIR}/python_env"
VENV_PYTHON = f"{VENV_DIR}/bin/python"

# ── Ensure system temporary directory exists BEFORE anything else runs ─────────
os.makedirs(f"{SCRIPT_DIR}/tmp", exist_ok=True)
os.chmod(f"{SCRIPT_DIR}/tmp", 0o777)

# ── Environment variables ────────────────────────────────────────────────────
os.environ["PYTHONUTF8"] = "1"
os.environ["PYTHONIOENCODING"] = "utf-8"
os.environ["TTS_CACHE"] = f"{SCRIPT_DIR}/models"
os.environ["TESSDATA_PREFIX"] = f"{SCRIPT_DIR}/models/tessdata"
os.environ["TMPDIR"] = "/tmp"  # Use standard /tmp for OS installers like Rustup
os.environ["SCRIPT_MODE"] = "native"  # Direct app.py to run in native environment mode
os.environ["MPLBACKEND"] = "Agg"

def run_cmd(cmd, desc, cwd=None):
    """Simple wrapper to run shell commands and display status."""
    print(f"➜ {desc}...")
    res = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=cwd or SCRIPT_DIR)
    if res.returncode == 0:
        print(f"{CHECK_MARK} Done.")
        return True
    else:
        print(f"{CROSS_MARK} Failed: {desc}\nError: {res.stderr.strip()}")
        return False

# ════════════════════════════════════════════════════════════════════════════
# FAST SETUP CHECKS & INSTALLS
# ════════════════════════════════════════════════════════════════════════════

print("⚡ Starting fast initialization...")

# --- Step 1: OS Dependencies (Fast Check) ---
print("\n--- Step 1: Checking OS-Level Dependencies ---")
if shutil.which("calibre") is None:
    print("Calibre not detected. Installing OS dependencies...")
    os_cmd = (
        "apt-get update -qq && "
        "apt-get install -y -qq python3.10 python3.10-venv python3.10-distutils libxcb-cursor0 libegl1 libopengl0 ffmpeg mediainfo nodejs espeak-ng sox tesseract-ocr tesseract-ocr-eng mecab libmecab-dev mecab-ipadic-utf8 && "
        "touch /etc/mecabrc && "
        "wget -nv -O- https://download.calibre-ebook.com/linux-installer.sh | sudo sh /dev/stdin"
    )
    run_cmd(os_cmd, "Installing system packages & Calibre", cwd="/content")
else:
    print(f"{CHECK_MARK} OS dependencies already present.")

# --- Step 2: Ensure Virtual Environment & UniDic Data Exists ---
print("\n--- Step 2: Checking Python Virtual Environment ---")

unidic_lite_path = f"{VENV_DIR}/lib/python3.10/site-packages/unidic_lite"

if not os.path.exists(VENV_PYTHON) or not os.path.exists(unidic_lite_path):
    print("Virtual environment or dictionary missing. Building/Repairing...")
    venv_cmd = (
        f"python3.10 -m venv --without-pip {VENV_DIR} && "
        f"curl -sSL https://bootstrap.pypa.io/get-pip.py | {VENV_PYTHON} && "
        f"{VENV_PYTHON} -m pip install -q --upgrade pip setuptools wheel packaging && "
        f"{VENV_PYTHON} -m pip install -q --upgrade llvmlite numba --only-binary=:all: && "
        f"sed -i '/ext\\/py\\/demucs/d' {SCRIPT_DIR}/requirements.txt && "
        f"{VENV_PYTHON} -m pip install -q --no-cache-dir -r {SCRIPT_DIR}/requirements.txt && "
        f"{VENV_PYTHON} -m pip install -e {SCRIPT_DIR}/ext/py/demucs --no-deps -q && "
        f"{VENV_PYTHON} -m pip install -q unidic-lite && "
        f"{VENV_PYTHON} -m pip uninstall -y unidic"
    )
    run_cmd(venv_cmd, "Rebuilding python_env and installing packages")
else:
    print(f"{CHECK_MARK} Python environment ready.")

# Force ensure broken `unidic` doesn't exist
run_cmd(f"{VENV_PYTHON} -m pip uninstall -y unidic", "Ensuring broken unidic is uninstalled to force fallback")

# --- Step 3: Run App ---
print("\n--- Step 3: Launch ebook2audiobook ---")
print("Starting the Gradio web interface with a public share link...")
os.environ["GRADIO_ALLOWED_PATHS"] = "/content/drive/MyDrive/Colab/ebook2audiobook,/content/ebook2audiobook"

try:
    get_ipython().system(
        f"cd {SCRIPT_DIR} && "
        f"GRADIO_ALLOWED_PATHS='/content/drive/MyDrive/Colab/ebook2audiobook,/content/ebook2audiobook' "
        f"VIRTUAL_ENV={VENV_DIR} {VENV_PYTHON} -u app.py --script_mode native --share"
    )
except Exception as e:
    print(f"{CROSS_MARK} Error starting app.py: {e}")